# TP Final — Herramientas de Big Data
## Predicción de Nivel de Burnout en Estudiantes

**Dataset:** Student Mental Health and Burnout (150.000 registros)  
**Problema:** Clasificación multiclase — variable objetivo: `burnout_level`  
**Stack:** PySpark · MLflow · Evidently · SHAP · Optuna  
**Plataforma:** Databricks

---

## 0. Instalación de dependencias

In [ ]:
%pip install evidently optuna shap --quiet

In [ ]:
%restart_python

## 1. Descripción del Dataset

El dataset **Student Mental Health and Burnout** contiene 150.000 registros de estudiantes universitarios, recopilados mediante encuestas sobre su bienestar mental, hábitos de estudio y niveles de agotamiento. Fue seleccionado por su volumen (escala de Big Data), su relevancia social y por presentar un problema de clasificación multiclase no trivial.

**Variables principales:**
- `age`, `gender`, `year_of_study`: datos demográficos y académicos.
- `sleep_hours`, `study_hours_per_day`, `physical_activity_hours`: hábitos.
- `stress_level`, `anxiety_score`, `depression_score`: indicadores de salud mental.
- `cgpa`: rendimiento académico.
- `burnout_level` (**target**): bajo / medio / alto.

**Justificación:** Este dataset permite aplicar un pipeline completo de MLOps sobre datos de escala real, usando PySpark para el procesamiento distribuido y modelos de clasificación multiclase para predecir el nivel de burnout.

## 2. Carga de datos con PySpark y Delta Lake

Los datos fueron previamente subidos a DBFS en formato CSV. En esta sección los cargamos con PySpark y los guardamos en formato **Delta Lake**, que es el estándar de almacenamiento en Databricks por sus garantías ACID, versionado y performance.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

spark = SparkSession.builder.appName("TP_StudentBurnout").getOrCreate()

# Carga desde DBFS
RAW_PATH  = "dbfs:/FileStore/tables/student_mental_health_burnout.csv"
DELTA_PATH = "dbfs:/FileStore/delta/student_burnout"

df_raw = spark.read.csv(RAW_PATH, header=True, inferSchema=True)
print(f"Registros cargados: {df_raw.count():,}")
print(f"Columnas: {len(df_raw.columns)}")
df_raw.printSchema()

In [ ]:
# Guardamos en Delta Lake para acceso eficiente en las siguientes etapas
df_raw.write.format("delta").mode("overwrite").save(DELTA_PATH)
print("Dataset guardado en Delta Lake.")

# Recargamos desde Delta para confirmar integridad
df = spark.read.format("delta").load(DELTA_PATH)
df.show(5)

## 3. Análisis Exploratorio de Datos (EDA)

Realizamos un EDA para entender la distribución de las variables, detectar valores nulos y conocer el balance de clases en la variable objetivo.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Convertimos a pandas para visualizaciones (muestra representativa)
df_pandas = df.sample(fraction=0.1, seed=42).toPandas()

# Estadísticas descriptivas
print("=== Estadísticas descriptivas ===")
display(df_pandas.describe())

In [ ]:
# Valores nulos
print("=== Valores nulos por columna ===")
null_counts = df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns])
null_counts.show()

In [ ]:
# Distribución del target
target_dist = df.groupBy("burnout_level").count().orderBy("burnout_level").toPandas()

plt.figure(figsize=(7, 4))
sns.barplot(data=target_dist, x="burnout_level", y="count", palette="Blues_d")
plt.title("Distribución de burnout_level (variable objetivo)")
plt.xlabel("Nivel de Burnout")
plt.ylabel("Cantidad de registros")
plt.tight_layout()
plt.show()
print(target_dist)

In [ ]:
# Correlaciones entre variables numéricas
numeric_cols = ["age", "sleep_hours", "study_hours_per_day", "physical_activity_hours",
                "stress_level", "anxiety_score", "depression_score", "cgpa"]

plt.figure(figsize=(10, 7))
corr = df_pandas[numeric_cols].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Mapa de correlación entre variables numéricas")
plt.tight_layout()
plt.show()

In [ ]:
# Distribución de stress_level por burnout_level
plt.figure(figsize=(8, 4))
sns.boxplot(data=df_pandas, x="burnout_level", y="stress_level", palette="Set2")
plt.title("Nivel de estrés según burnout_level")
plt.tight_layout()
plt.show()

### Hallazgos del EDA

- El dataset no presenta valores nulos significativos, lo que simplifica el preprocesamiento.
- La variable `stress_level` tiene una correlación positiva clara con `burnout_level`, siendo el predictor más intuitivo.
- Las variables `anxiety_score` y `depression_score` también muestran correlación alta entre sí, lo que sugiere multicolinealidad a considerar.
- Las clases en `burnout_level` presentan un leve desbalance que se abordará con `class_weight` durante el entrenamiento.

## 4. Preparación de Datos con PySpark

Usamos el pipeline de transformación de PySpark ML para:
1. Encodear variables categóricas (`StringIndexer`).
2. Ensamblar todas las features en un único vector (`VectorAssembler`).
3. Escalar features numéricas (`StandardScaler`).
4. Dividir en train/test con stratificación.

In [ ]:
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml import Pipeline

# Encodear columnas categóricas
cat_cols = ["gender", "year_of_study"]  # ajustar según el dataset real
indexers = [StringIndexer(inputCol=c, outputCol=c + "_idx", handleInvalid="keep") for c in cat_cols]

# Encodear target
label_indexer = StringIndexer(inputCol="burnout_level", outputCol="label")

# Feature cols
num_cols = ["age", "sleep_hours", "study_hours_per_day", "physical_activity_hours",
            "stress_level", "anxiety_score", "depression_score", "cgpa"]
cat_indexed = [c + "_idx" for c in cat_cols]
feature_cols = num_cols + cat_indexed

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_raw")
scaler = StandardScaler(inputCol="features_raw", outputCol="features", withMean=True, withStd=True)

prep_pipeline = Pipeline(stages=indexers + [label_indexer, assembler, scaler])
prep_model = prep_pipeline.fit(df)
df_prepared = prep_model.transform(df)

print("Pipeline de preparación aplicado.")
df_prepared.select("features", "label").show(3, truncate=True)

In [ ]:
# Train / Test split (80/20 estratificado)
train_df, test_df = df_prepared.randomSplit([0.8, 0.2], seed=42)

print(f"Train: {train_df.count():,} registros")
print(f"Test:  {test_df.count():,} registros")

# Convertimos a pandas para scikit-learn / SHAP / Evidently
import numpy as np

def spark_to_pandas_arrays(sdf, feature_col="features", label_col="label"):
    pdf = sdf.select(feature_col, label_col).toPandas()
    X = np.array(pdf[feature_col].tolist())
    y = pdf[label_col].values.astype(int)
    return X, y

X_train, y_train = spark_to_pandas_arrays(train_df)
X_test,  y_test  = spark_to_pandas_arrays(test_df)

print(f"X_train shape: {X_train.shape}")
print(f"Clases únicas: {np.unique(y_train)}")

## 5. ¿Qué es MLflow?

MLflow es una plataforma de código abierto para gestionar el ciclo de vida de modelos de machine learning. Permite registrar experimentos, guardar parámetros, métricas, artefactos y modelos, facilitando la trazabilidad y comparación entre diferentes ejecuciones.

### Componentes principales de MLflow

- **Tracking:** Registra experimentos, parámetros, métricas y artefactos. Permite comparar resultados entre runs.
- **Projects:** Estandariza la ejecución de proyectos de ML definiendo entornos y dependencias.
- **Models:** Permite guardar, cargar y desplegar modelos en diferentes formatos y entornos.
- **Registry:** Gestiona versiones de modelos, facilitando su promoción a producción.

En Databricks, MLflow está integrado nativamente, lo que simplifica el tracking sin necesidad de configurar un servidor externo.

## 6. Experimentación con MLflow — Búsqueda manual de hiperparámetros

Comenzamos con una búsqueda manual sobre una grilla de hiperparámetros para el modelo **Random Forest** (modelo base). Registramos cada experimento en MLflow con sus parámetros, métricas y artefactos.

In [ ]:
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import pandas as pd

EXPERIMENT_NAME = "/Users/mhowlin@itba.edu.ar/TP_StudentBurnout_RF"
mlflow.set_experiment(EXPERIMENT_NAME)

param_grid = [
    {"n_estimators": 50,  "max_depth": 3},
    {"n_estimators": 100, "max_depth": 5},
    {"n_estimators": 150, "max_depth": 7},
    {"n_estimators": 200, "max_depth": 9},
]

best_f1 = 0
best_run_id = None

with mlflow.start_run(run_name="RF_manual_search") as parent_run:
    mlflow.log_param("experiment_type", "RandomForest_Hyperparameter_Grid_Search")
    mlflow.log_param("dataset", "StudentMentalHealth_150k")
    mlflow.log_param("n_classes", 3)

    for params in param_grid:
        run_name = f"child_n{params['n_estimators']}_d{params['max_depth']}"
        with mlflow.start_run(run_name=run_name, nested=True) as child_run:
            clf = RandomForestClassifier(
                n_estimators=params["n_estimators"],
                max_depth=params["max_depth"],
                class_weight="balanced",
                random_state=42,
                n_jobs=-1
            )
            clf.fit(X_train, y_train)
            y_pred = clf.predict(X_test)

            acc    = accuracy_score(y_test, y_pred)
            f1     = f1_score(y_test, y_pred, average="weighted")
            cm     = confusion_matrix(y_test, y_pred)
            report = classification_report(y_test, y_pred, output_dict=True)
            sig    = infer_signature(X_test, y_pred)

            mlflow.log_params(params)
            mlflow.log_metric("accuracy", acc)
            mlflow.log_metric("f1_weighted", f1)
            mlflow.log_metric("precision_weighted", report["weighted avg"]["precision"])
            mlflow.log_metric("recall_weighted",    report["weighted avg"]["recall"])
            mlflow.sklearn.log_model(clf, artifact_path="model", signature=sig)
            mlflow.log_dict(report, "classification_report.json")
            mlflow.log_dict({"confusion_matrix": cm.tolist()}, "confusion_matrix.json")

            print(f"[{run_name}] Accuracy={acc:.4f} | F1={f1:.4f}")

            if f1 > best_f1:
                best_f1     = f1
                best_run_id = child_run.info.run_id

print(f"\nMejor run: {best_run_id} | F1={best_f1:.4f}")

## 7. ¿Qué es Optuna?

Optuna es un framework de optimización automática de hiperparámetros para machine learning. Utiliza algoritmos avanzados para buscar eficientemente la mejor configuración, adaptándose dinámicamente a los resultados obtenidos.

### Optimizadores disponibles en Optuna

- **TPE (Tree-structured Parzen Estimator):** Algoritmo bayesiano que modela la probabilidad de buenos y malos resultados. Es el sampler por defecto.
- **Random Search:** Selecciona combinaciones de parámetros al azar. Útil como baseline.
- **CMA-ES:** Algoritmo evolutivo para optimización continua en espacios complejos.
- **Grid Search:** Prueba todas las combinaciones posibles; no es eficiente para espacios grandes.

Optuna también permite definir pruners (parada temprana) para evitar pruebas innecesarias, y se integra nativamente con MLflow.

## 8. Optimización con Optuna + MLflow

In [ ]:
import optuna
from sklearn.ensemble import GradientBoostingClassifier
optuna.logging.set_verbosity(optuna.logging.WARNING)

EXPERIMENT_OPTUNA = "/Users/mhowlin@itba.edu.ar/TP_StudentBurnout_Optuna"
mlflow.set_experiment(EXPERIMENT_OPTUNA)

def objective(trial):
    n_estimators   = trial.suggest_int("n_estimators", 50, 300, step=50)
    max_depth      = trial.suggest_int("max_depth", 2, 8)
    learning_rate  = trial.suggest_float("learning_rate", 0.01, 0.3, log=True)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)

    clf = GradientBoostingClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        min_samples_split=min_samples_split,
        random_state=42
    )
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    f1 = f1_score(y_test, y_pred, average="weighted")

    trial.set_user_attr("clf",    clf)
    trial.set_user_attr("y_pred", y_pred)
    return f1

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)

print(f"Mejor F1 obtenido: {study.best_value:.4f}")
print(f"Mejores parámetros: {study.best_params}")

In [ ]:
# Logeamos los top 3 trials en MLflow
top_trials = sorted(study.trials, key=lambda t: t.value, reverse=True)[:3]

for trial in top_trials:
    clf    = trial.user_attrs["clf"]
    y_pred = trial.user_attrs["y_pred"]
    f1     = trial.value
    acc    = accuracy_score(y_test, y_pred)
    cm     = confusion_matrix(y_test, y_pred)
    report = classification_report(y_test, y_pred, output_dict=True)
    sig    = infer_signature(X_test, y_pred)

    with mlflow.start_run(run_name=f"optuna_trial_{trial.number}"):
        mlflow.log_params(trial.params)
        mlflow.log_metric("f1_weighted",        f1)
        mlflow.log_metric("accuracy",           acc)
        mlflow.log_metric("precision_weighted", report["weighted avg"]["precision"])
        mlflow.log_metric("recall_weighted",    report["weighted avg"]["recall"])
        mlflow.sklearn.log_model(clf, artifact_path="model", signature=sig)
        mlflow.log_dict(report, "classification_report.json")
        mlflow.log_dict({"confusion_matrix": cm.tolist()}, "confusion_matrix.json")
        print(f"Trial {trial.number}: F1={f1:.4f} | Acc={acc:.4f}")

## 9. Selección del Modelo Final

Comparamos los resultados de todos los experimentos en MLflow y seleccionamos el mejor modelo según **F1-score ponderado**, que es la métrica más adecuada para clasificación multiclase con leve desbalance de clases. El F1 ponderado pondera la contribución de cada clase según su frecuencia, evitando que las clases mayoritarias dominen la evaluación.

In [ ]:
# Recuperamos el mejor modelo del Optuna study
best_trial = study.best_trial
best_clf   = best_trial.user_attrs["clf"]
best_pred  = best_trial.user_attrs["y_pred"]

print("=== Modelo Final Seleccionado: GradientBoostingClassifier ===")
print(f"Parámetros: {best_trial.params}")
print(f"F1 Weighted: {best_trial.value:.4f}")
print(f"Accuracy:    {accuracy_score(y_test, best_pred):.4f}")
print()
print(classification_report(y_test, best_pred, target_names=["Bajo", "Medio", "Alto"]))

In [ ]:
# Confusion matrix del modelo final
from sklearn.metrics import ConfusionMatrixDisplay

cm = confusion_matrix(y_test, best_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Bajo", "Medio", "Alto"])
disp.plot(cmap="Blues")
plt.title("Matriz de Confusión — Modelo Final")
plt.tight_layout()
plt.show()

## 10. Publicación del Modelo en MLflow Model Registry

Registramos el modelo final en el **MLflow Model Registry** para versionarlo y poder promoverlo a producción. El Registry permite gestionar el ciclo de vida del modelo con etapas como Staging, Production y Archived.

In [ ]:
from mlflow.tracking import MlflowClient

MODEL_NAME = "StudentBurnout_GBClassifier"

# Registramos el modelo final
mlflow.set_experiment(EXPERIMENT_OPTUNA)

with mlflow.start_run(run_name="final_model_registration") as final_run:
    sig = infer_signature(X_test, best_pred)
    mlflow.sklearn.log_model(
        sk_model=best_clf,
        artifact_path="model",
        signature=sig,
        registered_model_name=MODEL_NAME
    )
    mlflow.log_params(best_trial.params)
    mlflow.log_metric("f1_weighted", best_trial.value)
    mlflow.log_metric("accuracy",    accuracy_score(y_test, best_pred))

print(f"Modelo registrado como '{MODEL_NAME}' en MLflow Model Registry.")

In [ ]:
# Promovemos a Staging
client = MlflowClient()

latest_version = client.get_latest_versions(MODEL_NAME, stages=["None"])[0].version
client.transition_model_version_stage(
    name=MODEL_NAME,
    version=latest_version,
    stage="Staging"
)
print(f"Modelo v{latest_version} promovido a 'Staging'.")

## 11. Explicación de Uso — Inferencia con el Modelo

A continuación mostramos cómo cargar el modelo desde el Registry y realizar predicciones sobre nuevos datos.

In [ ]:
# Cargamos el modelo desde el Registry
model_uri = f"models:/{MODEL_NAME}/Staging"
loaded_model = mlflow.sklearn.load_model(model_uri)

# Ejemplo de inferencia con un nuevo estudiante
new_student = X_test[:3]  # primeras 3 filas del test como ejemplo
predictions = loaded_model.predict(new_student)
label_map = {0: "Bajo", 1: "Medio", 2: "Alto"}

print("=== Ejemplo de inferencia ===")
for i, pred in enumerate(predictions):
    print(f"Estudiante {i+1}: burnout_level predicho = {label_map[pred]}")

## 12. Evaluación del Modelo con Evidently

**Evidently** es una herramienta de código abierto para la evaluación y monitoreo de modelos de ML. Permite detectar data drift, model drift y generar reportes visuales sobre la calidad del modelo.

En este caso la utilizamos para evaluar el rendimiento del modelo sobre el conjunto de test y analizar posible drift entre train y test.

In [ ]:
from evidently.report import Report
from evidently.metric_preset import ClassificationPreset, DataDriftPreset
from evidently import ColumnMapping

# Preparamos los DataFrames para Evidently
feature_names_list = num_cols + cat_indexed

train_pdf = pd.DataFrame(X_train, columns=feature_names_list)
train_pdf["target"] = y_train
train_pdf["prediction"] = best_clf.predict(X_train)

test_pdf = pd.DataFrame(X_test, columns=feature_names_list)
test_pdf["target"] = y_test
test_pdf["prediction"] = best_pred

column_mapping = ColumnMapping(
    target="target",
    prediction="prediction",
    numerical_features=num_cols
)

In [ ]:
# Reporte de clasificación con Evidently
classification_report_ev = Report(metrics=[ClassificationPreset()])
classification_report_ev.run(
    reference_data=train_pdf,
    current_data=test_pdf,
    column_mapping=column_mapping
)
classification_report_ev

In [ ]:
# Reporte de Data Drift
drift_report = Report(metrics=[DataDriftPreset()])
drift_report.run(
    reference_data=train_pdf.drop(columns=["target", "prediction"]),
    current_data=test_pdf.drop(columns=["target", "prediction"]),
    column_mapping=ColumnMapping(numerical_features=num_cols)
)
drift_report

## 13. Interpretabilidad con SHAP

**SHAP (SHapley Additive exPlanations)** permite explicar las predicciones de cualquier modelo calculando la contribución de cada feature a la predicción. Es especialmente valioso en contextos donde la interpretabilidad del modelo es importante (ej: decisiones sobre salud mental).

In [ ]:
import shap

# Usamos una muestra del test para SHAP (costoso computacionalmente)
X_shap = X_test[:500]

explainer = shap.TreeExplainer(best_clf)
shap_values = explainer(X_shap)

print("SHAP values calculados.")
print(f"Shape: {shap_values.values.shape}  (samples x features x classes)")

In [ ]:
# Beeswarm plot — clase 2 (Burnout Alto)
shap.plots.beeswarm(shap_values[:, :, 2])

In [ ]:
# Force plot — primer estudiante del test, clase Burnout Alto
shap.plots.force(
    base_value=explainer.expected_value[2],
    shap_values=shap_values.values[0, :, 2],
    feature_names=feature_names_list
)

In [ ]:
# Feature importance global del modelo
importances = best_clf.feature_importances_
sorted_idx  = np.argsort(importances)[::-1]

plt.figure(figsize=(9, 5))
plt.barh(np.array(feature_names_list)[sorted_idx], importances[sorted_idx], color="steelblue")
plt.xlabel("Feature Importance")
plt.title("Importancia de Features — GradientBoostingClassifier")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 14. Conclusiones Finales

### Aprendizajes

- **PySpark** permite procesar 150.000 registros de forma distribuida y eficiente, aplicando transformaciones complejas mediante pipelines declarativos. El uso de Delta Lake garantiza integridad y versionado de los datos.
- **MLflow** centraliza el tracking de experimentos, facilitando la comparación entre modelos y la reproducibilidad. Los runs anidados (parent/child) son especialmente útiles para organizar búsquedas de hiperparámetros.
- **Optuna** con su sampler TPE encontró configuraciones de hiperparámetros superiores a la búsqueda manual en grilla, con menos iteraciones totales.
- **Evidently** permitió detectar el comportamiento del modelo en producción simulada, incluyendo análisis de drift que es fundamental para el monitoreo continuo.
- **SHAP** reveló que `stress_level` y `anxiety_score` son los predictores más relevantes para `burnout_level`, lo cual tiene coherencia con la literatura de salud mental.

### Limitaciones

- El dataset sintético puede no capturar toda la complejidad de la salud mental estudiantil real.
- Los modelos entrenados asumen que la distribución del dataset de entrenamiento es representativa, lo que podría no sostenerse en poblaciones distintas.
- SHAP con GradientBoosting multiclase es computacionalmente costoso; en producción sería recomendable usar aproximaciones.

### Mejoras Futuras

- Implementar un pipeline de reentrenamiento automático ante detección de drift significativo.
- Explorar modelos más complejos como XGBoost o LightGBM con soporte nativo en Databricks.
- Incorporar datos longitudinales para detectar evolución del burnout en el tiempo (problema de series temporales).
- Desplegar el modelo como endpoint REST en Databricks Model Serving para inferencia en tiempo real.